In [1]:
import torch
from torch import nn
from torch.nn import functional

In [3]:
from dataclasses import dataclass

@dataclass
class ModelConfig:
    vocab_size:int
    d_model:int
    max_seq_len: int # context length
    num_heads:int
    num_layers:int
    dropout:float

In [4]:
model_config = ModelConfig(
    vocab_size=44,
    d_model=32,
    max_seq_len=5,
    num_heads=4,
    num_layers=2,
    dropout=0.2
)

In [9]:
class EmbeddingLayer(nn.Module):
    def __init__(self,model_config:ModelConfig):
        super().__init__()

        """
        Always remember that embedding layer is not actually a neural laer with any activation function ,
        its just an 2d matrix lookup the matrix size is (vocab_size,d_model) but but but the d_model here is lets one array
         so that array is already have learnable parameter means in our case that 32 dim arrays have the values from the training 
         for e.g:
         token_embeding[1] will be the embeding of token id 1 which is 32 dim embedding array
        """
        self.token_embeddings=nn.Embedding(model_config.vocab_size,model_config.d_model)
        self.pos_embeddings = nn.Embedding(model_config.max_seq_len,model_config.d_model)

    def forward(self,token_ids):
        pos_ids = torch.arange(token_ids.shape[1],device=token_ids.device)
        embedding = self.token_embeddings(token_ids) + self.pos_embeddings(pos_ids)
        return embedding


In [5]:
class MultiHeadAttention(nn.Module):
    def __init__(self,model_config:ModelConfig):
        super().__init__()

        self.config = model_config
        # initialize q,k,v
        self.w_q = nn.Linear(self.config.d_model,self.config.d_model,bias=False)
        self.w_k = nn.Linear(self.config.d_model,self.config.d_model,bias=False)
        self.w_v = nn.Linear(self.config.d_model,self.config.d_model,bias=False)

        self.register_buffer(
                            "casual_mask", 
                            torch.tril(
                                torch.ones(
                                    self.config.max_seq_len, 
                                    self.config.max_seq_len
                                    )
                                )
                        )
        
        self.head_dim = self.config.d_model//self.config.num_heads
        

        self.w_o = nn.Linear(self.config.d_model,self.config.d_model,bias=False)

    def forward(self,x):
        # here x is the embeding and not raw x  and b is batch size, t is the current sequence length and d_model is hidden dimension
        # shape of x -> (b,t,d_model)
        q = self.w_q(x) # (b,t,d_model)
        k = self.w_k(x) # (b,t,d_model)
        v = self.w_v(x) # (b,t,d_model)

        q = torch.reshape(q,(x.shape[0],x.shape[1],self.config.num_heads,self.head_dim))  # (b,t,num_head,head_dim)
        k = torch.reshape(k,(x.shape[0],x.shape[1],self.config.num_heads,self.head_dim))  # (b,t,num_head,head_dim)
        v = torch.reshape(v,(x.shape[0],x.shape[1],self.config.num_heads,self.head_dim))  # (b,t,num_head,head_dim)


        q = q.transpose(1,2) # (b,num_head,t,head_dim)
        k = k.transpose(1,2) # (b,num_head,t,head_dim)
        v = v.transpose(1,2) # (b,num_head,t,head_dim)

        scores = q @ k.transpose(-2,-1) / (self.head_dim ** 0.5) # (b,num_head,t,t)
        mask = self.casual_mask[:x.shape[1],:x.shape[1]] # (t,t)
        masked_score = scores.masked_fill(mask==0,float("-inf")) # (b,num_head,t,t)
        attention_weights = torch.softmax(masked_score,dim = -1) # (b,num_head,t,t)
        context_vector = attention_weights @ v # (b,num_head,t,head_dim)

        context_vector = context_vector.transpose(1,2) # (b,t,num_head,head_dim) 
        context_vector = context_vector.contiguous().view(x.shape) # (b,t,d_model)
        return self.w_o(context_vector) #(b,t,d_model) which is same as the input embedding 


In [6]:
class FeedForward(nn.Module):
    def __init__(self,model_config:ModelConfig):
        super().__init__()
        self.config = model_config
        self.model = nn.Sequential(
            nn.Linear(self.config.d_model,4*self.config.d_model),
            nn.GELU(),
            nn.Linear(4*self.config.d_model,self.config.d_model)
        )

    def forward(self,x):
        return self.model(x)

In [8]:
class DecoderBlock(nn.Module):
    def __init__(self,model_config:ModelConfig):
        super().__init__()

        self.config = model_config
        self.attention_layer = MultiHeadAttention(model_config)
        self.ffn = FeedForward(model_config)
        self.ln_1 =  nn.LayerNorm(model_config.d_model) # before attention layer
        self.ln_2 =  nn.LayerNorm(model_config.d_model) # before feed forward layer

    def forward(self,x):
        x =  x+self.attention_layer(self.ln_1(x))
        x = x+ self.ffn(self.ln_2(x))
        return x



In [56]:
class Transformer(nn.Module):
    def __init__(self,model_config:ModelConfig):
        super().__init__()
        self.config = model_config
        self.embedding_layer = EmbeddingLayer(model_config)
        self.drop = nn.Dropout(model_config.dropout)
        self.block = nn.ModuleList([DecoderBlock(model_config) for _ in range(model_config.num_layers)])
        self.ln_f = nn.LayerNorm(model_config.d_model)
        self.lm_head = nn.Linear(model_config.d_model,model_config.vocab_size,bias=False)
        self.lm_head.weight = self.embedding_layer.token_embeddings.weight

        self.apply(self._init_weights)

    def _init_weights(self,module):
        if isinstance(module, nn.Linear):
            module.weight = torch.nn.init.normal_(module.weight,mean=0.0,std=0.02)
            if module.bias is not None:
                module.bias = torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            module.weight = nn.init.normal_(module.weight,mean=0,std=0.02)

        
    def forward(self,x,targets=None):
        embedding = self.embedding_layer(x)
        out = self.drop(embedding)
        for i in range(self.config.num_layers):
            out = self.block[i](out)

        out_norm = self.ln_f(out)
        logits = self.lm_head(out_norm)
        loss = None
        if targets is not None:
            loss = functional.cross_entropy(
                logits.view(-1,self.config.vocab_size),
                targets.view(-1))
        return logits,loss

    def generate(self,idx,max_new_tokens):
        for _ in range(max_new_tokens):
            idx_cond = idx[:,-self.config.max_seq_len:] #index conditional
            logits,_ = self(idx_cond)
            last_token = logits[:,-1,:]
            probs = functional.softmax(last_token,dim=-1)
            curr_id = torch.multinomial(probs,num_samples=1)
            idx = torch.cat((idx, curr_id), dim=1)

        return idx

In [57]:
model = Transformer(model_config)

In [58]:
model

Transformer(
  (embedding_layer): EmbeddingLayer(
    (token_embeddings): Embedding(44, 32)
    (pos_embeddings): Embedding(5, 32)
  )
  (drop): Dropout(p=0.2, inplace=False)
  (block): ModuleList(
    (0-1): 2 x DecoderBlock(
      (attention_layer): MultiHeadAttention(
        (w_q): Linear(in_features=32, out_features=32, bias=False)
        (w_k): Linear(in_features=32, out_features=32, bias=False)
        (w_v): Linear(in_features=32, out_features=32, bias=False)
        (w_o): Linear(in_features=32, out_features=32, bias=False)
      )
      (ffn): FeedForward(
        (model): Sequential(
          (0): Linear(in_features=32, out_features=128, bias=True)
          (1): GELU(approximate='none')
          (2): Linear(in_features=128, out_features=32, bias=True)
        )
      )
      (ln_1): LayerNorm((32,), eps=1e-05, elementwise_affine=True, bias=True)
      (ln_2): LayerNorm((32,), eps=1e-05, elementwise_affine=True, bias=True)
    )
  )
  (ln_f): LayerNorm((32,), eps=1e-05, e

In [59]:
x = torch.randint(0,44,(2,5))
x 

tensor([[43,  1, 39, 39, 42],
        [18,  0, 42, 31, 25]])

In [60]:
logits,loss= model(x)

In [61]:
logits.shape

torch.Size([2, 5, 44])

In [62]:
loss

In [63]:
y = torch.randint(0,44,(2,5))
y

tensor([[11,  7, 18,  0, 28],
        [27, 30, 22, 17, 17]])

In [64]:
logits ,loss = model(x,y)

In [65]:
logits.shape

torch.Size([2, 5, 44])

In [66]:
loss

tensor(3.7941, grad_fn=<NllLossBackward0>)

In [67]:
total_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {total_params:,}")

Total parameters: 26,784


In [73]:
prompt = torch.ones((1,1),dtype=torch.long)

In [74]:
output_ids = model.generate(prompt,max_new_tokens=10)
print("Generated Token IDs:\n", output_ids)
print("Shape:", output_ids.shape)

Generated Token IDs:
 tensor([[ 1, 42, 37,  6, 42, 17, 29,  6,  8, 36, 27]])
Shape: torch.Size([1, 11])
